# Notebook 03 — Preprocessing + Sentiment Analysis
**Owner:** Person 3

**Input:** `../data/reddit/reddit_raw.csv` (from Notebook 02)

**Goal:** Clean Reddit text, score sentiment per post using VADER, and aggregate to a weekly sentiment index.

**Output:** `../data/processed/sentiment_weekly.csv`

**Columns:** `week | subreddit | avg_sentiment | post_count | avg_upvotes`

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
import matplotlib.pyplot as plt

nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)

nlp = spacy.load('en_core_web_sm')
vader = SentimentIntensityAnalyzer()

## 1. Load Reddit Data

In [ ]:
df = pd.read_csv("../data/reddit/reddit_raw.csv", parse_dates=["date"])
print(f"Loaded {len(df)} posts")
print(df["subreddit"].value_counts())
df.head()

## 2. Build Full Text Field
Combine title + body + top comments for richer sentiment signal.

In [ ]:
def combine_text(row) -> str:
    parts = [
        str(row.get("title", "") or ""),
        str(row.get("body", "") or ""),
        str(row.get("top_comments", "") or "")
    ]
    return " ".join(parts)

df["full_text"] = df.apply(combine_text, axis=1)
print("Sample full text:")
print(df["full_text"].iloc[0][:300])

## 3. Text Preprocessing

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def preprocess(text: str) -> str:
    """Lowercase, remove URLs/special chars, lemmatize, remove stopwords."""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove special characters and digits (keep letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Lowercase
    text = text.lower().strip()
    # Lemmatize with spaCy
    doc = nlp(text, disable=["ner", "parser"])
    tokens = [
        token.lemma_ for token in doc
        if token.lemma_ not in STOP_WORDS and len(token.lemma_) > 2
    ]
    return " ".join(tokens)

df["text_clean"] = df["full_text"].apply(preprocess)
print("Preprocessing done")
df[["full_text", "text_clean"]].head(2)

## 4. Sentiment Scoring with VADER
VADER is designed for social media text — handles slang, caps, punctuation well.

We score on raw `full_text` (not preprocessed) — VADER relies on punctuation/capitalization for signal.

In [ ]:
def score_sentiment(text: str) -> float:
    """Return VADER compound score: -1 (very negative) to +1 (very positive)."""
    return vader.polarity_scores(text)["compound"]

df["sentiment_score"] = df["full_text"].apply(score_sentiment)

# Label for reference
def label(score):
    if score >= 0.05: return "positive"
    if score <= -0.05: return "negative"
    return "neutral"

df["sentiment_label"] = df["sentiment_score"].apply(label)

print(df["sentiment_label"].value_counts())
df[["title", "sentiment_score", "sentiment_label"]].head()

## 5. Aggregate to Weekly Sentiment

In [ ]:
df["week"] = df["date"].dt.to_period("W")

sentiment_weekly = (
    df.groupby(["week", "subreddit"])
    .agg(
        avg_sentiment=("sentiment_score", "mean"),
        post_count=("post_id", "count"),
        avg_upvotes=("upvotes", "mean")
    )
    .reset_index()
)

# Also compute combined (all subreddits) row
combined = (
    df.groupby("week")
    .agg(
        avg_sentiment=("sentiment_score", "mean"),
        post_count=("post_id", "count"),
        avg_upvotes=("upvotes", "mean")
    )
    .reset_index()
)
combined["subreddit"] = "ALL"

sentiment_weekly = pd.concat([sentiment_weekly, combined], ignore_index=True)
print(f"Weekly sentiment rows: {len(sentiment_weekly)}")
sentiment_weekly.head(10)

## 6. Save Output

In [ ]:
OUTPUT_PATH = "../data/processed/sentiment_weekly.csv"
sentiment_weekly.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

## 7. Quick Visualization

In [ ]:
all_weekly = sentiment_weekly[sentiment_weekly["subreddit"] == "ALL"].copy()
all_weekly["week_dt"] = all_weekly["week"].dt.to_timestamp()
all_weekly = all_weekly.sort_values("week_dt")

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(all_weekly["week_dt"], all_weekly["avg_sentiment"], color="steelblue", label="Avg Sentiment")
ax1.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax1.set_ylabel("Avg VADER Sentiment Score")
ax1.set_title("Weekly Reddit Sentiment — Hardware Communities")

ax2 = ax1.twinx()
ax2.bar(all_weekly["week_dt"], all_weekly["post_count"], alpha=0.2, color="orange", label="Post Count")
ax2.set_ylabel("Post Count")

plt.tight_layout()
plt.savefig("../data/processed/sentiment_timeline.png", dpi=150)
plt.show()